# Ejercicio 6

In [ ]:
import numpy as np
from sklearn.neighbors import LocalOutlierFactor
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder, StandardScaler, OrdinalEncoder, TargetEncoder, FunctionTransformer
from sklearn.neighbors import LocalOutlierFactor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier


## Inciso 1

In [1]:
%pip install gdown

#carpeta con los datos de Google Drive
import sys
!{sys.executable} -m gdown --folder 1GOJ63mZ6qZGdbb8v5v6bcy3fsbLs8yju

#cargar el dataset en un DataFrame

import pandas as pd
df = pd.read_pickle('data/house_prices.pkl')
df.head()


Note: you may need to restart the kernel to use updated packages.


'd:\existencia\facu' is not recognized as an internal or external command,
operable program or batch file.


,SalePrice,MSZoning,Neighborhood,LotFrontage,LotArea,OverallCond,YearBuilt,FullBath,BedroomAbvGr,GarageQual,GarageArea,PoolArea,PoolQC,Fence
0,87500,RL,NAmes,NaN,8544,4,1949,2,2,TA,400,0,NaN,NaN
1,164990,RL,CollgCr,65.0,8767,5,2005,2,3,TA,400,0,NaN,NaN
2,144000,RM,CollgCr,NaN,4435,5,2003,1,1,TA,420,0,NaN,NaN
3,307000,rL,Somerst,75.0,10084,5,2004,2,3,TA,636,0,NaN,NaN
4,130500,RL,Sawyer,NaN,13517,8,1976,2,3,TA,475,0,NaN,NaN


Todas las operaciones que utilizen medidas "estadisticas" o utilizen datos de otras observaciones del dataset, introducen data leakeage si se hacen con los datos de testeo.

Corregir errores de tipeo/ inconsistencias en nomenclatura, se lo hacemos al conjunto completo, ya que no fuga ningun dato.

El borrado de duplicados deberia de hacerse con el dataset completo: Si el dato que esta duplicado, cae en el conjunto de entrenamiento y en el conjunto de testeo, el modelo podria memorizar la respuesta.

In [5]:
df['SalePrice']=df['SalePrice'].astype(float)

#reemplazamos los "NA" por un nan
df['LotFrontage']=df['LotFrontage'].replace('NA',np.nan)
#por las dudas convertimos todos a numericos
df['LotFrontage']=df['LotFrontage'].astype(float)

print("Tipo final:", df['LotFrontage'].dtype)
print("Nulos reales:", df['LotFrontage'].isna().sum())

Tipo final: float64
Nulos reales: 264


In [ ]:
from sklearn.model_selection import train_test_split

# Dividir solo el conjunto de datos X
X_train, X_test = train_test_split(df, test_size=0.3, random_state=42)


## Inciso 2

In [ ]:
num_cols = df.select_dtypes(include=['float64', 'int64']).dropna().copy()
columnas_texto=df.select_dtypes(include=['object']).columns
#iteramos para eliminar los espacios
for col in columnas_texto:
    df[col]=df[col].str.upper().str.strip()

# 2. dropna procedural sobre variables numéricas en Train (alineando target)
valid_num_mask = X_train[num_cols].notna().all(axis=1)
X_train_nonan = X_train.loc[valid_num_mask]

# 3. Escalado robusto temporal para alimentar LOF
robust_temp = RobustScaler()
X_train_num_scaled = robust_temp.fit_transform(X_train_nonan[num_cols])

# 4. Detección y filtrado de outliers con LOF (1: inlier, -1: outlier)
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
inlier_mask = (lof.fit_predict(X_train_num_scaled) == 1)

# Filtrado final de Train
X_train_final = X_train_nonan.iloc[inlier_mask]

# 1. Definición de grupos de columnas
col_zoning = ['MSZoning']
col_neighborhood = ['Neighborhood']
col_garage_qual = ['GarageQual']
col_pool = ['PoolQC']
col_fence = ['Fence']

# 2. Sub-pipelines específicos

# MSZoning: Imputar moda y OHE
pipe_zoning = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Neighborhood: Target Encoding con validación interna y suavizado
pipe_neighborhood = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target_enc', TargetEncoder(smooth='auto', cv=5))
])

# GarageQual: 'NA' es 'Sin Garaje' -> Orden: NA < Po < Fa < TA < Gd < Ex
garage_order = ['NA', 'PO', 'FA', 'TA', 'GD', 'EX']
pipe_garage = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
    ('ordinal', OrdinalEncoder(
        categories=[garage_order], 
        handle_unknown='use_encoded_value', 
        unknown_value=-1
    ))
])

# PoolQC: Binarizador (NaN -> 0, Cualquier valor -> 1)
def binarize_sparsity(df_or_array):
    return (~pd.isna(df_or_array)).astype(int)

pipe_pool = Pipeline([
    ('binarizer', FunctionTransformer(binarize_sparsity))
])

# Fence: Imputar 'None' a los nulos y OHE
pipe_fence = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Integración en ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('zoning', pipe_zoning, col_zoning),
        ('neighborhood', pipe_neighborhood, col_neighborhood),
        ('garage', pipe_garage, col_garage_qual),
        ('pool', pipe_pool, col_pool),
        ('fence', pipe_fence, col_fence)
    ],
    remainder='passthrough'
)